## 評価したい<br>
・まずは動画をフレームごとに分けて、画像をフレームごとに保存する。またそれに対してフレームIDを付与する<br>
・フレームIDごとにどのラベルなのかを考えて、付与していく。<br>
└この場合、顔検出精度（人数を正しく検出したか）とラベル精度（各顔の順序含め完全一致したか）の二つを評価する<br>

In [12]:
import cv2
import numpy as np
from collections import defaultdict
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import pandas as pd
import matplotlib.pyplot as plt
import os
from deepface import DeepFace

### 動画を１フレームごとに切り出す

In [5]:
def extract_frames(video_path, output_dir):
    # ディレクトリが存在しなければ作成
    os.makedirs(output_dir, exist_ok=True)

    print(f"処理対象: {video_path}")
    print(f"ファイル存在確認: {os.path.exists(video_path)}")

    video = cv2.VideoCapture(video_path)

    # ビデオが正しく開かれたか確認
    if not video.isOpened():
        print("エラー：動画ファイルが開けません")
    else:
        fps = int(video.get(cv2.CAP_PROP_FPS))
        total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
        print(f"FPS: {fps}, 総フレーム数: {total_frames}, 解像度: {width}×{height}")
        
        frame_count = 0
        saved_frames = []

        while True:
            success, frame = video.read() 
            if not success:
                break  # 動画の終了

            # 1秒ごとにフレームを保存
            if frame_count % fps == 0:
                frame_time_sec = int(frame_count / fps)
                frame_file = os.path.join(output_dir, f"frame_{frame_time_sec:05d}.jpg")
                # リサイズなしで元のサイズのまま保存
                cv2.imwrite(frame_file, frame)
                saved_frames.append((frame_time_sec, frame_file))
            
            frame_count += 1

        video.release()
        print(f"✅ 処理完了！保存されたフレーム数: {len(saved_frames)}")

In [ ]:
output_dir = "./movie_frame"
video_path = "./movie/nogi_movie_001.mp4"

extract_frames(video_path, output_dir)

### 元画像データの特徴量をface_dbに格納

In [16]:
def get_deepface_db(base_path, model_name='Facenet'):
    face_db = {}
    target_folders = ["endou", "kaki","yumiki"]
    
    print("--- DeepFace 特徴量抽出開始 ---")
    for label in target_folders:
        dir_path = os.path.join(base_path, label)
        if not os.path.exists(dir_path): continue
        
        embeddings = []
        for img_name in os.listdir(dir_path):
            if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')): continue
            try:
                # 登録時は精度の高い検出器を使用
                objs = DeepFace.represent(img_path=os.path.join(dir_path, img_name), 
                                          model_name=model_name, 
                                          detector_backend='retinaface', 
                                          enforce_detection=False)
                for obj in objs:
                    embeddings.append(obj["embedding"])
            except:
                continue
        
        if embeddings:
            face_db[label] = np.mean(embeddings, axis=0)
            print(f"Registered: {label} ({len(embeddings)} images)")
            
    return face_db

### 複数人ケースの3つの評価指標

| 指標 | 説明 | 例 |
|------|------|-----|
| **顔検出精度** | 正解の人数を正しく検出できたか | 正解2人 → 検出2人 = ○ |
| **セット照合精度** | フレーム内の全顔セットが完全一致するか | {A,B} == {A,B} = ○ |
| **個別ラベル精度** | 各顔の識別ラベルが正確か（人数違う場合も評価可能） | 正解[A,B] vs 予測[A,C] = 50% |


In [ ]:
def evaluate_multi_person_detailed(test_images_dir="./movie_frame", 
                                   db_base_path="./", 
                                   model_name='Facenet', 
                                   threshold=0.40):
    """
    複数人テスト画像の詳細評価（3つの精度指標）
    
    1. 顔検出精度: 正解数と完全一致 → 人数が合っているか
    2. セット照合精度: セット一致度 → 正確な識別ができているか  
    3. 個別ラベル精度: ラベル一致度 → 各顔の識別スコア
    """
    face_db = get_deepface_db(db_base_path, model_name=model_name)
    
    anno_file = os.path.join(test_images_dir, "annotations.csv")
    if not os.path.exists(anno_file):
        print(f"⚠️ {anno_file} が見つかりません")
        return None, None
    
    anno_df = pd.read_csv(anno_file)
    
    detection_correct = 0  # 人数が正しい
    set_correct = 0        # セットが完全一致
    label_correct = 0      # 個別ラベルが一致
    
    detailed_results = []
    
    print("=== 複数人テスト画像 詳細評価 ===\n")
    print(f"{'Image':<20} {'正解':<15} {'予測':<15} {'検出':<5} {'セット':<5} {'ラベル':<6}")
    print("-" * 75)
    
    for _, row in anno_df.iterrows():
        img_name = row['image_name']
        img_path = os.path.join(test_images_dir, img_name)
        
        if not os.path.exists(img_path):
            continue
        
        gt_persons = sorted([p.strip() for p in str(row['persons']).split('|') if p.strip()])
        # personsの列を'|'で分割してリスト化し、空白を除去してソート
        
        try:
            objs = DeepFace.represent(img_path=img_path, 
                                     model_name=model_name, 
                                     detector_backend='retinaface', 
                                     enforce_detection=False)
            
            pred_persons = []
            
            for obj in objs:
                conf = obj.get("face_confidence", 0)
                if conf > 0.3:
                    current_feat = np.array(obj["embedding"])
                    
                    best_score = -1
                    best_label = "Unknown"
                    for label, target_feat in face_db.items():
                        score = np.dot(target_feat, current_feat) / \
                               (np.linalg.norm(target_feat) * np.linalg.norm(current_feat))
                        if score > best_score:
                            best_score = score
                            best_label = label
                    
                    is_match = best_score > (1 - threshold)
                    final_label = best_label if is_match else "Unknown"
                    pred_persons.append(final_label)
            
            pred_persons = sorted(pred_persons)
            
            # 3つの評価指標
            detection_ok = (len(pred_persons) == len(gt_persons))  # 人数一致
            set_ok = (set(pred_persons) == set(gt_persons))        # セット一致
            label_ok = (pred_persons == gt_persons)                # 順序含め完全一致
            
            if detection_ok:
                detection_correct += 1
            if set_ok:
                set_correct += 1
            if label_ok:
                label_correct += 1
            
            # 表示
            det_mark = "✓" if detection_ok else "✗"
            set_mark = "✓" if set_ok else "✗"
            label_mark = "✓" if label_ok else "✗"
            
            gt_str = '|'.join(gt_persons)
            pred_str = '|'.join(pred_persons)
            
            print(f"{img_name:<20} {gt_str:<15} {pred_str:<15} {det_mark:<5} {set_mark:<5} {label_mark:<6}")
            
            detailed_results.append({
                'image': img_name,
                'gt': gt_str,
                'pred': pred_str,
                'detection': detection_ok,
                'set_match': set_ok,
                'label_match': label_ok
            })
        
        except:
            print(f"{img_name:<20} {'Error':<15}")
    
    # 集計
    total = len(anno_df)
    print("\n" + "="*50)
    print(f"📊 集計結果 (計 {total} 画像)")
    print("="*50)
    print(f"顔検出精度 (人数一致):    {detection_correct}/{total} ({100*detection_correct/total:.1f}%)")
    print(f"セット照合精度 (順不同):    {set_correct}/{total} ({100*set_correct/total:.1f}%)")
    print(f"ラベル精度 (順序含む完全一致): {label_correct}/{total} ({100*label_correct/total:.1f}%)")
    
    summary_df = pd.DataFrame([{
        'detection_count': detection_correct,
        'detection_rate': 100 * detection_correct / total if total else 0.0,
        'set_count': set_correct,
        'set_rate': 100 * set_correct / total if total else 0.0,
        'label_count': label_correct,
        'label_rate': 100 * label_correct / total if total else 0.0,
    }])
    
    return pd.DataFrame(detailed_results), summary_df

# 使用例
detailed_results, summary_df = evaluate_multi_person_detailed()
summary_df.to_clipboard()
summary_df

--- DeepFace 特徴量抽出開始 ---
Registered: endou (15 images)
Registered: kaki (59 images)
Registered: yumiki (34 images)
=== 複数人テスト画像 詳細評価 ===

Image                正解              予測              検出    セット   ラベル   
---------------------------------------------------------------------------
frame_00000.jpg      endou           endou           ✓     ✓     ✓     
frame_00001.jpg      endou           endou           ✓     ✓     ✓     
frame_00002.jpg      endou           endou           ✓     ✓     ✓     
frame_00003.jpg      kaki            kaki            ✓     ✓     ✓     
frame_00004.jpg      kaki            kaki            ✓     ✓     ✓     
frame_00005.jpg      kaki            yumiki          ✓     ✗     ✗     
frame_00006.jpg      endou           endou           ✓     ✓     ✓     
frame_00007.jpg      endou           endou           ✓     ✓     ✓     
frame_00008.jpg      endou           endou           ✓     ✓     ✓     
frame_00009.jpg      endou           endou           ✓     ✓     ✓

In [19]:
detailed_results.to_clipboard()

### 📝 複数人ケースの評価フロー

**準備1: テスト画像セットの作成**
```
test_multi_person/
├── single_endou.jpg        （1人のみ）
├── single_kaki.jpg         （1人のみ）
├── pair_001.jpg            （2人一緒）
├── pair_002.jpg            （2人一緒）
└── annotations.csv
```

**準備2: annotations.csv の作成**
```csv
image_name,persons
single_endou.jpg,endou
single_kaki.jpg,kaki
pair_001.jpg,endou|kaki
pair_002.jpg,endou|kaki
```

**実行: 詳細評価で3つの指標を確認**
```python
detailed_results = evaluate_multi_person_detailed("./test_multi_person")
```

**出力例:**
```
顔検出精度 (人数一致):    4/4 (100.0%)
セット照合精度 (順不同):    4/4 (100.0%)
ラベル精度 (順序含む完全一致): 3/4 (75.0%)
```


face_recognitionによる予測

face rekognitionのライブラリを当初は使う予定だった。正しいかのような問題が起こった<br>
・pip installするときにdlibライブラリを合わせてインストールする必要があり、C++の環境が必要でインストールが大変<br>
・pip installできてもライブラリをimportする際に、別ライブラリを入れる必要があるエラーが起こった<br>
利用までがかなり煩雑であるため、Deepfaceを使う方針へ転換

In [ ]:
import face_recognition
import cv2

# Open the input movie file
input_movie = cv2.VideoCapture("hamilton_clip.mp4")
length = int(input_movie.get(cv2.CAP_PROP_FRAME_COUNT))

# Create an output movie file (make sure resolution/frame rate matches input video!)
fourcc = cv2.VideoWriter_fourcc(*'XVID')
output_movie = cv2.VideoWriter('output.avi', fourcc, 29.97, (640, 360))

# Load some sample pictures and learn how to recognize them.
lmm_image = face_recognition.load_image_file("lin-manuel-miranda.png")
lmm_face_encoding = face_recognition.face_encodings(lmm_image)[0]

al_image = face_recognition.load_image_file("alex-lacamoire.png")
al_face_encoding = face_recognition.face_encodings(al_image)[0]

known_faces = [
    lmm_face_encoding,
    al_face_encoding
]

# Initialize some variables
face_locations = []
face_encodings = []
face_names = []
frame_number = 0

while True:
    # Grab a single frame of video
    ret, frame = input_movie.read()
    frame_number += 1

    # Quit when the input video file ends
    if not ret:
        break

    # Convert the image from BGR color (which OpenCV uses) to RGB color (which face_recognition uses)
    rgb_frame = frame[:, :, ::-1]

    # Find all the faces and face encodings in the current frame of video
    face_locations = face_recognition.face_locations(rgb_frame)
    face_encodings = face_recognition.face_encodings(rgb_frame, face_locations)

    face_names = []
    for face_encoding in face_encodings:
        # See if the face is a match for the known face(s)
        match = face_recognition.compare_faces(known_faces, face_encoding, tolerance=0.50)

        # If you had more than 2 faces, you could make this logic a lot prettier
        # but I kept it simple for the demo
        name = None
        if match[0]:
            name = "Lin-Manuel Miranda"
        elif match[1]:
            name = "Alex Lacamoire"

        face_names.append(name)

    # Label the results
    for (top, right, bottom, left), name in zip(face_locations, face_names):
        if not name:
            continue

        # Draw a box around the face
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 0, 255), 2)

        # Draw a label with a name below the face
        cv2.rectangle(frame, (left, bottom - 25), (right, bottom), (0, 0, 255), cv2.FILLED)
        font = cv2.FONT_HERSHEY_DUPLEX
        cv2.putText(frame, name, (left + 6, bottom - 6), font, 0.5, (255, 255, 255), 1)

    # Write the resulting image to the output video file
    print("Writing frame {} / {}".format(frame_number, length))
    output_movie.write(frame)

# All done!
input_movie.release()
cv2.destroyAllWindows()

Googleだと以下のエラーが出た<br>
TypeError: 'NoneType' object is not iterable<br>
おそらく、スクレイピングをしようと思った際にパース（解析）処理の例外エラーが発生したものと思われる。そのため今回はBingを使うのが良いと判断

In [9]:
crawler = GoogleImageCrawler(storage={'root_dir' : './data_google'})
crawler.crawl(keyword = keyword, max_num = photo_num)

2026-02-24 22:58:49,562 - INFO - icrawler.crawler - start crawling...
2026-02-24 22:58:49,564 - INFO - icrawler.crawler - starting 1 feeder threads...
2026-02-24 22:58:49,567 - INFO - feeder - thread feeder-001 exit
2026-02-24 22:58:49,569 - INFO - icrawler.crawler - starting 1 parser threads...
2026-02-24 22:58:49,573 - INFO - icrawler.crawler - starting 1 downloader threads...
2026-02-24 22:58:50,250 - INFO - parser - parsing result page https://www.google.com/search?q=%E8%B3%80%E5%96%9C%E9%81%A5%E9%A6%99&ijn=0&start=0&tbs=&tbm=isch
Exception in thread parser-001:
Traceback (most recent call last):
  File "C:\Users\真希\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "C:\Users\真希\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\真希\Downloads\nogi_movie_naming\venv\Lib\site-pa